In [28]:
SEED = 42

In [29]:
from pathlib import Path
from helpers.data.ticker_loader import load_tickers

TICKERS_FILE = Path("tickers.txt")
if not TICKERS_FILE.exists():
    TICKERS_FILE = Path("NN_Trading_project/tickers.txt")

# None        → all tickers
# ["AAPL", …] → explicit list
# 42          → random sample of 42 (seeded by SEED)
TICKER_SUBSET = 400

TICKERS = load_tickers(TICKERS_FILE, subset=TICKER_SUBSET, seed=SEED)
print(f"Using {len(TICKERS)} tickers")


Using 400 tickers


In [30]:
import pandas as pd
import datetime

# ============================================================
# DATE SETTINGS
# ============================================================

# --- Date Range ---
INTERVAL   = "1d"
START_DATE = pd.Timestamp("2025-01-01")           # inclusive
END_DATE   = pd.Timestamp(datetime.date.today())  # inclusive

# --- Train / Val / Test Split ---
# Train: Date <= TRAIN_END_DATE
# Val:   (TRAIN_END_DATE, VAL_END_DATE]
# Test:  Date > VAL_END_DATE
TRAIN_END_DATE = pd.Timestamp("2025-11-30")
VAL_END_DATE   = pd.Timestamp("2026-01-20")

# --- Exclusion Window (applied during featurize) ---
EXCLUDE_START_DATE = "2018-09-01"  # inclusive
EXCLUDE_END_DATE   = "2023-06-30"  # inclusive


# --- Exclusion Window (Features) ---
REBUILD_FEATURE_CACHE = True  # set True to recompute when features/WINDOW change


In [31]:
# ============================================================
# HYPERPARAMETERS - MODEL & TRAINING
# ============================================================

# --- Trading / Labeling ---
# Buy at market OPEN using prior day's OHLCV + current day's OPEN only.
HORIZON_BARS     = 0         # look-ahead bars for profit hit
PROFIT_THRESHOLD = 1 / 100   # 1% profit target vs current day's OPEN
STOP_LOSS        = -3 / 100  # -2% stop loss vs current day's OPEN
WINDOW           = 30        # look-back window for features

# --- Training ---
MAX_EPOCHS    = 50
PATIENCE      = 10   # early-stopping patience

BUY_THRESHOLD = 0.5  # minimum probability to classify as BUY

# --- Plotting ---
PLOT_GRAPH1_EPOCH_INTERVAL = 1
PLOT_GRAPH2_BATCH_INTERVAL = 5
PLOT_GRAPH3_EPOCH_INTERVAL = 1

# --- Optimizer ---
BATCH_SIZE   = 64

# --- Model Architecture ---
HIDDEN_SIZES = [64]  # e.g. [1024, 512] or [256, 128]

# --- Data / DataLoader ---
SPLIT_FRAC  = 0.85  # train fraction (time-based)
NUM_WORKERS = 16    # DataLoader workers (0 for debugging)


In [32]:
from pathlib import Path
from helpers.data.date_config_manager import check_and_refresh_date_config

_current_config = {
    "INTERVAL":           str(INTERVAL),
    "START_DATE":         str(START_DATE.date()),
    "END_DATE":           str(END_DATE.date()),
    "TRAIN_END_DATE":     str(TRAIN_END_DATE.date()),
    "VAL_END_DATE":       str(VAL_END_DATE.date()),
    "EXCLUDE_START_DATE": str(EXCLUDE_START_DATE),
    "EXCLUDE_END_DATE":   str(EXCLUDE_END_DATE),
    "TICKER_SUBSET":      str(TICKER_SUBSET),
}

check_and_refresh_date_config(
    current_config = _current_config,
    config_path    = Path.cwd() / "date_config.txt",
    stocks_dir     = Path.cwd() / "dataset" / "stocks",
)


Date config changed — clearing cached CSVs for a fresh download.
  Previous config:
    INTERVAL: 1d
    START_DATE: 2025-01-01
    END_DATE: 2026-02-28
    TRAIN_END_DATE: 2025-11-30
    VAL_END_DATE: 2026-01-20
    EXCLUDE_START_DATE: 2018-09-01
    EXCLUDE_END_DATE: 2023-06-30
    TICKER_SUBSET: 200 <-- CHANGED
  Deleted: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/dataset/stocks
  Updated: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/date_config.txt


True

In [33]:
# Step 1: download NASDAQ data into the *dataset* directory using yfinance (DAILY)
from pathlib import Path
import pandas as pd
from helpers.data.data_downloader import download_tickers

# Root directory where yfinance CSVs will be stored

data_root = Path.cwd() / "dataset"
stocks_dir = data_root / "stocks"

# call helper
_summary = download_tickers(
    tickers= TICKERS,
    start=   START_DATE,
    end=     END_DATE,
    interval=INTERVAL,
    out_dir= stocks_dir,
)


 OK   :: AA rows=289 | downloaded 1/400
 OK   :: AAOI rows=289 | downloaded 2/400
 OK   :: AAPL rows=289 | downloaded 3/400
 OK   :: AB rows=289 | downloaded 4/400
 OK   :: ABEV rows=289 | downloaded 5/400
 OK   :: ABNB rows=289 | downloaded 6/400
 OK   :: ACI rows=289 | downloaded 7/400
 OK   :: ADI rows=289 | downloaded 8/400
 OK   :: ADSK rows=289 | downloaded 9/400
 OK   :: AEO rows=289 | downloaded 10/400
 OK   :: AEP rows=289 | downloaded 11/400
 OK   :: AFRM rows=289 | downloaded 12/400
 OK   :: AGNC rows=289 | downloaded 13/400
 OK   :: AJG rows=289 | downloaded 14/400
 OK   :: ALB rows=289 | downloaded 15/400


$ALE: possibly delisted; no timezone found

1 Failed download:
['ALE']: possibly delisted; no timezone found


 OK   :: ALGN rows=289 | downloaded 16/400
 OK   :: ALKS rows=289 | downloaded 17/400
 OK   :: ALL rows=289 | downloaded 18/400
 OK   :: ALNY rows=289 | downloaded 19/400
 OK   :: AMD rows=289 | downloaded 20/400
 OK   :: AMN rows=289 | downloaded 21/400
 OK   :: AMRZ rows=173 | downloaded 22/400
 OK   :: ANET rows=289 | downloaded 23/400
 OK   :: AON rows=289 | downloaded 24/400
 OK   :: APA rows=289 | downloaded 25/400
 OK   :: APD rows=289 | downloaded 26/400
 OK   :: APGE rows=289 | downloaded 27/400
 OK   :: APP rows=289 | downloaded 28/400
 OK   :: AR rows=289 | downloaded 29/400
 OK   :: ARCB rows=289 | downloaded 30/400
 OK   :: ARE rows=289 | downloaded 31/400
 OK   :: ARWR rows=289 | downloaded 32/400
 OK   :: ASML rows=289 | downloaded 33/400
 OK   :: ATXS rows=265 | downloaded 34/400
 OK   :: AVA rows=289 | downloaded 35/400
 OK   :: AVAV rows=289 | downloaded 36/400
 OK   :: AVB rows=289 | downloaded 37/400
 OK   :: AVGO rows=289 | downloaded 38/400
 OK   :: AVTR rows=289 

$LAZR: possibly delisted; no timezone found

1 Failed download:
['LAZR']: possibly delisted; no timezone found


 OK   :: LCID rows=289 | downloaded 206/400
 OK   :: LCTX rows=289 | downloaded 207/400
 OK   :: LFUS rows=289 | downloaded 208/400
 OK   :: LI rows=289 | downloaded 209/400
 OK   :: LMND rows=289 | downloaded 210/400
 OK   :: LOGI rows=289 | downloaded 211/400
 OK   :: LSCC rows=289 | downloaded 212/400
 OK   :: LVS rows=289 | downloaded 213/400
 OK   :: MAA rows=289 | downloaded 214/400
 OK   :: MAC rows=289 | downloaded 215/400
 OK   :: MAN rows=289 | downloaded 216/400
 OK   :: MANH rows=289 | downloaded 217/400
 OK   :: MAR rows=289 | downloaded 218/400
 OK   :: MARA rows=289 | downloaded 219/400
 OK   :: MAS rows=289 | downloaded 220/400
 OK   :: MCHP rows=289 | downloaded 221/400
 OK   :: MDLZ rows=289 | downloaded 222/400
 OK   :: MELI rows=289 | downloaded 223/400
 OK   :: MGY rows=289 | downloaded 224/400
 OK   :: MKC rows=289 | downloaded 225/400
 OK   :: MLI rows=289 | downloaded 226/400
 OK   :: MMYT rows=289 | downloaded 227/400
 OK   :: MOH rows=289 | downloaded 228/400


$SPR: possibly delisted; no timezone found

1 Failed download:
['SPR']: possibly delisted; no timezone found


 OK   :: SPXC rows=289 | downloaded 322/400
 OK   :: SRAD rows=289 | downloaded 323/400
 OK   :: SRPT rows=289 | downloaded 324/400
 OK   :: SSB rows=289 | downloaded 325/400
 OK   :: STE rows=289 | downloaded 326/400
 OK   :: STNE rows=289 | downloaded 327/400
 OK   :: STWD rows=289 | downloaded 328/400
 OK   :: STZ rows=289 | downloaded 329/400
 OK   :: SWBI rows=289 | downloaded 330/400
 OK   :: SYNA rows=289 | downloaded 331/400
 OK   :: SYY rows=289 | downloaded 332/400
 OK   :: TCBI rows=289 | downloaded 333/400
 OK   :: TCEHY rows=289 | downloaded 334/400
 OK   :: TCOM rows=289 | downloaded 335/400
 OK   :: TDW rows=289 | downloaded 336/400
 OK   :: TEAM rows=289 | downloaded 337/400
 OK   :: TECK rows=289 | downloaded 338/400
 OK   :: TER rows=289 | downloaded 339/400
 OK   :: TFC rows=289 | downloaded 340/400
 OK   :: TFX rows=289 | downloaded 341/400
 OK   :: TJX rows=289 | downloaded 342/400
 OK   :: TMUS rows=289 | downloaded 343/400
 OK   :: TRIN rows=289 | downloaded 344/

# Building Features + Labels

In [34]:
import pickle
import shutil
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from helpers.feature.feature_builder import precompute_and_cache, FEATURE_COLS
from helpers.data.dataset import StockDatasetSafe, is_cache_valid

# ── Paths ────────────────────────────────────────────────────────────────────
root       = Path.cwd() / "dataset"
stocks_dir = root / "stocks"
assert stocks_dir.exists(), f"Missing: {stocks_dir}"

files     = sorted(stocks_dir.glob("*.csv"))
cache_dir = Path.cwd() / f".feature_cache_forward_return_w{WINDOW}"
cache_dir.mkdir(parents=True, exist_ok=True)
scaler_path = cache_dir / "scaler.pkl"
index_path  = cache_dir / "index.pkl"

# ── Clear old cache if requested ─────────────────────────────────────────────
if REBUILD_FEATURE_CACHE:
    stale = [p for p in Path.cwd().glob(".feature*") if p.exists()]
    for p in stale:
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        else:
            p.unlink(missing_ok=True)
        print(f"[Cache] Removed: {p}")
    if not stale:
        print("[Cache] No stale .feature* paths found.")
    time.sleep(1)

# ── Build or load cache ───────────────────────────────────────────────────────
# Validates that scaler, index, AND every referenced .npz file all exist.
cache_ready = is_cache_valid(scaler_path, index_path)
if not cache_ready and cache_dir.exists():
    print("[Cache] Stale / incomplete cache detected — wiping cache dir.")
    shutil.rmtree(cache_dir)

# Ensure cache_dir exists before building or loading
cache_dir.mkdir(parents=True, exist_ok=True)

if REBUILD_FEATURE_CACHE or not cache_ready:
    scaler, index = precompute_and_cache(
        files           = files,
        window          = WINDOW,
        cache_dir       = cache_dir,
        scaler_path     = scaler_path,
        index_path      = index_path,
        horizon_bars    = HORIZON_BARS,
        train_end_date  = TRAIN_END_DATE,
        val_end_date    = VAL_END_DATE,
        profit_threshold= PROFIT_THRESHOLD,
        stop_loss       = STOP_LOSS,
        exclude_start   = EXCLUDE_START_DATE,
        exclude_end     = EXCLUDE_END_DATE,
    )
else:
    print("[Cache] Using existing feature cache.")
    with open(scaler_path, "rb") as f: scaler = pickle.load(f)
    with open(index_path,  "rb") as f: index  = pickle.load(f)

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = StockDatasetSafe(index, scaler, "train")
val_ds   = StockDatasetSafe(index, scaler, "val")
test_ds  = StockDatasetSafe(index, scaler, "test")

# ── DataLoaders ───────────────────────────────────────────────────────────────
_pin    = torch.cuda.is_available()
_kwargs = dict(
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    pin_memory  = _pin,
    persistent_workers = (NUM_WORKERS > 0),
    prefetch_factor    = 2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, shuffle=True,  **_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kwargs)

# ── Sanity check ──────────────────────────────────────────────────────────────
xb, yb = next(iter(train_loader))
print(f"Train samples : {len(train_ds):,}")
print(f"Val   samples : {len(val_ds):,}")
print(f"Test  samples : {len(test_ds):,}")
print(f"X batch : {xb.shape}  {xb.dtype}")
print(f"y batch : {yb.shape}  {yb.dtype}")
print(f"Feature count : {len(FEATURE_COLS)} base + {WINDOW-1} lags = {len(FEATURE_COLS) * WINDOW}")

[Cache] Removed: /scratch/temp_/NN_Trading_Bot_Private/NN_Trading_project/.feature_cache_forward_return_w30
[Cache] Precomputing features (leakage-safe splits)...
[Cache] (1/397) AA.csv
[Cache] (2/397) AAOI.csv
[Cache] (3/397) AAPL.csv
[Cache] (4/397) AB.csv
[Cache] (5/397) ABEV.csv
[Cache] (6/397) ABNB.csv
[Cache] (7/397) ACI.csv
[Cache] (8/397) ADI.csv
[Cache] (9/397) ADSK.csv
[Cache] (10/397) AEO.csv
[Cache] (11/397) AEP.csv
[Cache] (12/397) AFRM.csv
[Cache] (13/397) AGNC.csv
[Cache] (14/397) AJG.csv
[Cache] (15/397) ALB.csv
[Cache] (16/397) ALGN.csv
[Cache] (17/397) ALKS.csv
[Cache] (18/397) ALL.csv
[Cache] (19/397) ALNY.csv
[Cache] (20/397) AMD.csv
[Cache] (21/397) AMN.csv
[Cache] (22/397) AMRZ.csv
[Cache] (23/397) ANET.csv
[Cache] (24/397) AON.csv
[Cache] (25/397) APA.csv
[Cache] (26/397) APD.csv
[Cache] (27/397) APGE.csv
[Cache] (28/397) APP.csv
[Cache] (29/397) AR.csv
[Cache] (30/397) ARCB.csv
[Cache] (31/397) ARE.csv
[Cache] (32/397) ARWR.csv
[Cache] (33/397) ASML.csv
[Cache] 

# XGBoost

In [ ]:

import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import log_loss

# ── Helpers ───────────────────────────────────────────────────────────────────

def loader_to_numpy(loader):
    """Flatten a torch DataLoader into (X, y) numpy arrays."""
    Xs, ys = [], []
    for xb, yb in loader:
        Xs.append(xb.numpy())
        ys.append(yb.numpy())
    return np.concatenate(Xs), np.concatenate(ys)


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, ntree: int) -> np.ndarray:
    """Version-safe probability prediction from a native XGBoost Booster."""
    dmat = xgb.DMatrix(X)
    try:
        return booster.predict(dmat, iteration_range=(0, ntree)).astype(np.float32)
    except TypeError:
        return booster.predict(dmat, ntree_limit=ntree).astype(np.float32)


def buy_metrics(y_true, probs, threshold: float) -> dict:
    """Confusion matrix + accuracy + P(success | BUY) at a given threshold."""
    pred = (np.asarray(probs).ravel() >= threshold).astype(np.int64)
    y    = np.asarray(y_true).ravel().astype(np.int64)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "acc":         100.0 * (tp + tn) / max(1, len(y)),
        "buy_success": 100.0 * tp / max(1, tp + fp),
    }


def logloss_curve(booster: xgb.Booster, X: np.ndarray, y: np.ndarray, iters: np.ndarray) -> np.ndarray:
    """Compute log-loss at each iteration count (post-training, for test curve)."""
    dmat = xgb.DMatrix(X)
    y = y.astype(np.int64)
    out = []
    for k in iters:
        try:
            p = booster.predict(dmat, iteration_range=(0, int(k)))
        except TypeError:
            p = booster.predict(dmat, ntree_limit=int(k))
        out.append(log_loss(y, p))
    return np.asarray(out, dtype=np.float64)


def evaluate_and_save_test(
    booster: xgb.Booster,
    ntree: int,
    X_test: np.ndarray,
    y_test: np.ndarray,
    threshold: float,
    save_csv: str = "test_predictions_full.csv",
    test_tickers: list = None,
    pct_changes: list = None,
):
    """Full test evaluation: saves CSV and prints metrics."""
    probs = predict_probs_booster(booster, X_test, ntree)
    pred  = (probs >= threshold).astype(np.int64)
    ytrue = y_test.astype(np.int64)

    df = pd.DataFrame({
        "idx":          np.arange(len(ytrue), dtype=np.int64),
        "prob_buy":     probs.astype(np.float32),
        "pred":         pred,
        "actual":       ytrue,
        "correct":      pred == ytrue,
        "decision":     np.where(pred == 1, "BUY", "NO-BUY"),
        "actual_label": np.where(ytrue == 1, "BUY", "NO-BUY"),
    })
    if test_tickers is not None and len(test_tickers) == len(df):
        df["ticker"] = test_tickers
    if pct_changes is not None and len(pct_changes) == len(df):
        df["pct_change"] = pct_changes
    df.to_csv(save_csv, index=False)
    print(f"Saved test predictions → {save_csv}")

    m  = buy_metrics(ytrue, probs, threshold)
    tp, fp, tn, fn = m["tp"], m["fp"], m["tn"], m["fn"]
    n  = max(1, len(ytrue))
    print(f"P(success | BUY): {m['buy_success']:.2f}%  |  acc: {m['acc']:.2f}%")

    summary_df = pd.DataFrame({
        "category":             ["BUY_success_TP", "BUY_fail_FP", "NO_BUY_success_TN", "NO_BUY_fail_FN"],
        "count":                [tp, fp, tn, fn],
        "pct_of_all_%":         [100*tp/n, 100*fp/n, 100*tn/n, 100*fn/n],
        "pct_given_decision_%": [100*tp/max(1,tp+fp), 100*fp/max(1,tp+fp),
                                 100*tn/max(1,tn+fn), 100*fn/max(1,tn+fn)],
    })
    display(summary_df)

    return df, summary_df


# ── 1. Build arrays from loaders ──────────────────────────────────────────────

X_train, y_train = loader_to_numpy(train_loader)
X_val,   y_val   = loader_to_numpy(val_loader)
X_test,  y_test  = loader_to_numpy(test_loader)

num_pos = float((y_train == 1).sum())
num_neg = float((y_train == 0).sum())
scale_pos_weight = num_neg / max(1.0, num_pos)

print(f"Train {X_train.shape}  pos={int(num_pos)}  neg={int(num_neg)}")
print(f"Val   {X_val.shape}    pos={int((y_val==1).sum())}  neg={int((y_val==0).sum())}")
print(f"Test  {X_test.shape}   pos={int((y_test==1).sum())}  neg={int((y_test==0).sum())}")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

# ── 2. Define params ──────────────────────────────────────────────────────────

NUM_BOOST_ROUND       = 10000
EARLY_STOPPING_ROUNDS = max(5, PATIENCE * 5)

params = {
    "max_depth":        3,
    "eta":              0.01,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "alpha":            3.0,
    "lambda":           2.0,
    "scale_pos_weight": scale_pos_weight,
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
}

# ── 3. Train ──────────────────────────────────────────────────────────────────

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

evals_result = {}
print(f"Training: max_rounds={NUM_BOOST_ROUND}, early_stop={EARLY_STOPPING_ROUNDS}")
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    evals_result=evals_result,
    verbose_eval=100,
)

best_ntree = int(booster.best_iteration + 1) if booster.best_iteration is not None else NUM_BOOST_ROUND
print(f"\nBest iteration: {best_ntree}")

# ── 4. Final metrics ──────────────────────────────────────────────────────────

train_ll = np.asarray(evals_result["train"]["logloss"])
val_ll   = np.asarray(evals_result["val"]["logloss"])

probs_train = predict_probs_booster(booster, X_train, best_ntree)
probs_val   = predict_probs_booster(booster, X_val,   best_ntree)
probs_test  = predict_probs_booster(booster, X_test,  best_ntree)
tr = buy_metrics(y_train, probs_train, BUY_THRESHOLD)
vl = buy_metrics(y_val,   probs_val,   BUY_THRESHOLD)
te = buy_metrics(y_test,  probs_test,  BUY_THRESHOLD)

print(f"Train  logloss={log_loss(y_train,probs_train):.6f}  acc={tr['acc']:.2f}%  P(success|BUY)={tr['buy_success']:.2f}%")
print(f"Val    logloss={log_loss(y_val,  probs_val  ):.6f}  acc={vl['acc']:.2f}%  P(success|BUY)={vl['buy_success']:.2f}%")
print(f"Test   logloss={log_loss(y_test, probs_test ):.6f}  acc={te['acc']:.2f}%  P(success|BUY)={te['buy_success']:.2f}%")

# ── 5. Training curves ────────────────────────────────────────────────────────

iters_full = np.arange(1, len(train_ll) + 1)

# ── 6. Save model ────────────────────────────────────────────────────────────

bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("Saved → best_model_xgb.pkl")


Train (48887, 1320)  pos=22704  neg=26183
Val   (13260, 1320)    pos=6278  neg=6982
Test  (10468, 1320)   pos=5649  neg=4819
scale_pos_weight = 1.1532
Training: max_rounds=10000, early_stop=50
[0]	train-logloss:0.69271	val-logloss:0.69271
[100]	train-logloss:0.67086	val-logloss:0.67333
[200]	train-logloss:0.66387	val-logloss:0.67001
[300]	train-logloss:0.65903	val-logloss:0.66886
[400]	train-logloss:0.65500	val-logloss:0.66871


### Evaluate on Test Data 

In [ ]:

import pickle
import numpy as np
import pandas as pd

with open(index_path, "rb") as f:
    index = pickle.load(f)

test_tickers, pct_changes = [], []

for entry in index:
    data   = np.load(entry["cache_file"])
    ticker = entry["ticker"]

    test_start = entry.get("test_start", entry.get("val_cut", None))
    if test_start is None:
        raise KeyError(f"Index entry for {ticker} missing 'test_start' / 'val_cut'.")
    test_start = int(test_start)

    y_full     = data["y"]
    dates_full = data.get("dates", None)
    n_test     = int(len(y_full)) - test_start
    if n_test <= 0:
        continue

    test_tickers.extend([ticker] * n_test)

    if dates_full is None:
        pct_changes.extend([np.nan] * n_test)
        continue

    test_dates = pd.to_datetime(dates_full[test_start:]).tz_localize(None)
    csv_path   = stocks_dir / f"{ticker}.csv"
    try:
        df_csv = pd.read_csv(csv_path)
        if "Date" not in df_csv.columns:
            pct_changes.extend([np.nan] * n_test)
            continue

        df_csv["Date"] = pd.to_datetime(df_csv["Date"], errors="coerce")
        df_csv = (df_csv.dropna(subset=["Date"])
                        .sort_values("Date")
                        .set_index("Date"))
        df_csv = df_csv[~df_csv.index.duplicated(keep="last")]

        sub         = df_csv.reindex(test_dates)
        sub["Open"]  = pd.to_numeric(sub.get("Open"),  errors="coerce")
        sub["Close"] = pd.to_numeric(sub.get("Close"), errors="coerce")
        pct = ((sub["Close"] - sub["Open"]) / sub["Open"] * 100.0).to_numpy(dtype=np.float64)

        if len(pct) < n_test:
            pct = np.concatenate([pct, np.full(n_test - len(pct), np.nan)])
        else:
            pct = pct[:n_test]
        pct_changes.extend(pct.tolist())

    except Exception:
        pct_changes.extend([np.nan] * n_test)

assert len(test_tickers) == len(X_test), f"ticker mismatch: {len(test_tickers)} vs {len(X_test)}"
assert len(pct_changes)  == len(X_test), f"pct_change mismatch: {len(pct_changes)} vs {len(X_test)}"

df_test_preds, df_test_summary = evaluate_and_save_test(
    booster=booster,
    ntree=best_ntree,
    X_test=X_test,
    y_test=y_test,
    threshold=BUY_THRESHOLD,
    save_csv="test_predictions_full.csv",
    test_tickers=test_tickers,
    pct_changes=pct_changes,
)


Saved test predictions → test_predictions_full.csv
P(success | BUY): 57.31%  |  acc: 55.24%


,category,count,pct_of_all_%,pct_given_decision_%
0,BUY_success_TP,2034,38.610478,57.311919
1,BUY_fail_FP,1515,28.758542,42.688081
2,NO_BUY_success_TN,876,16.628702,50.959860
3,NO_BUY_fail_FN,843,16.002278,49.040140


# Optuna Hyperparameter Search (XGBoost)

In [ ]:

# ── Optuna: XGBoost Hyperparameter Search ─────────────────────────────────────

import numpy as np
import optuna
import xgboost as xgb
from sklearn.metrics import log_loss

optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_N_TRIALS     = 80
OPTUNA_EARLY_STOP   = max(10, PATIENCE * 5)
OPTUNA_MAX_ROUNDS   = 3000          # max boosting rounds per trial

dtrain_opt = xgb.DMatrix(X_train, label=y_train)
dval_opt   = xgb.DMatrix(X_val,   label=y_val)
dtest_opt  = xgb.DMatrix(X_test,  label=y_test)


def objective(trial: optuna.Trial) -> float:
    """Minimise validation log-loss."""
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "logloss",
        "tree_method":      "hist",
        "seed":             SEED,
        "scale_pos_weight": scale_pos_weight,
        # ── tuneable ────────────────────────────────────────────────────────
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "eta":              trial.suggest_float("eta", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "alpha":            trial.suggest_float("alpha", 0.0, 10.0),
        "lambda":           trial.suggest_float("lambda", 0.5, 10.0),
    }

    evals_res = {}
    bst = xgb.train(
        params=params,
        dtrain=dtrain_opt,
        num_boost_round=OPTUNA_MAX_ROUNDS,
        evals=[(dval_opt, "val")],
        early_stopping_rounds=OPTUNA_EARLY_STOP,
        evals_result=evals_res,
        verbose_eval=False,
    )

    best_iter   = int(bst.best_iteration + 1) if bst.best_iteration is not None else OPTUNA_MAX_ROUNDS
    val_logloss = evals_res["val"]["logloss"][best_iter - 1]

    trial.set_user_attr("best_ntree",   best_iter)
    trial.set_user_attr("val_logloss",  val_logloss)
    return val_logloss


# ── Run the study ─────────────────────────────────────────────────────────────

study = optuna.create_study(direction="minimize",
                             study_name="xgb_hparam_search",
                             sampler=optuna.samplers.TPESampler(seed=SEED))

print(f"Starting Optuna search: {OPTUNA_N_TRIALS} trials …")
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)

best_trial  = study.best_trial
best_params = best_trial.params
best_val_ll = best_trial.value
print(f"\nBest trial #{best_trial.number}  val_logloss={best_val_ll:.6f}")
print("Best params:", best_params)

# ── Retrain with best params on train+val ──────────────────────────────────────

final_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "tree_method":      "hist",
    "seed":             SEED,
    "scale_pos_weight": scale_pos_weight,
    **best_params,
}

# Use best_ntree from the winning trial (already tuned via early-stopping)
best_ntree_optuna = int(best_trial.user_attrs["best_ntree"])

dtrain_full = xgb.DMatrix(
    np.concatenate([X_train, X_val]),
    label=np.concatenate([y_train, y_val]),
)

print(f"\nRetraining on train+val for {best_ntree_optuna} rounds …")
booster = xgb.train(
    params=final_params,
    dtrain=dtrain_full,
    num_boost_round=best_ntree_optuna,
    verbose_eval=False,
)
best_ntree = best_ntree_optuna

# ── Evaluate on test ───────────────────────────────────────────────────────────

probs_test_optuna = predict_probs_booster(booster, X_test, best_ntree)
te_opt = buy_metrics(y_test, probs_test_optuna, BUY_THRESHOLD)
test_ll_val = log_loss(y_test, probs_test_optuna)

probs_train_optuna = predict_probs_booster(booster, X_train, best_ntree)
tr_opt = buy_metrics(y_train, probs_train_optuna, BUY_THRESHOLD)

print(f"\nTrain  acc={tr_opt['acc']:.2f}%  P(success|BUY)={tr_opt['buy_success']:.2f}%")
print(f"Test   acc={te_opt['acc']:.2f}%  P(success|BUY)={te_opt['buy_success']:.2f}%  logloss={test_ll_val:.6f}")

# ── Plots ──────────────────────────────────────────────────────────────────────

# 1. Optimisation history

# ── Save retrained model ──────────────────────────────────────────────────────
import joblib
bundle = {"booster": booster, "best_ntree": best_ntree}
joblib.dump(bundle, "best_model_xgb.pkl")
print("\nSaved → best_model_xgb.pkl")
print(f"Use BUY_THRESHOLD = {BUY_THRESHOLD} (unchanged; tune separately if needed)")


Starting Optuna search: 80 trials …


Best trial: 2. Best value: 0.668041:   8%|▊         | 6/80 [04:48<59:22, 48.14s/it]


[W 2026-02-28 10:25:10,550] Trial 6 failed with parameters: {'max_depth': 5, 'eta': 0.002870165242185818, 'subsample': 0.9847923138822793, 'colsample_bytree': 0.8650796940166687, 'min_child_weight': 19, 'gamma': 4.474136752138244, 'alpha': 5.978999788110851, 'lambda': 9.25780523271961} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/scratch/temp_/venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_85134/2411830559.py", line 39, in objective
    bst = xgb.train(
          ^^^^^^^^^^
  File "/scratch/temp_/venv/lib/python3.12/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/scratch/temp_/venv/lib/python3.12/site-packages/xgboost/training.py", line 200, in train
    bst.update(dtrain, iteration=i, fobj=obj)
  File "/scratch/temp_/venv/lib/python3.12/site

KeyboardInterrupt: 

# Eval Data Analysis

In [ ]:

# ── Val Data Analysis: trades-per-day ─────────────────────────────────────────

import numpy as np
import pandas as pd
import pickle
import joblib
import xgboost as xgb
import wandb

SELECTED_THRESHOLD = 0.9

with open(index_path, "rb") as f:
    index = pickle.load(f)
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

bundle     = joblib.load("best_model_xgb.pkl")
booster    = bundle["booster"]
best_ntree = int(bundle["best_ntree"])


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, ntree: int) -> np.ndarray:
    dmat = xgb.DMatrix(X)
    try:
        return booster.predict(dmat, iteration_range=(0, ntree)).astype(np.float32)
    except TypeError:
        return booster.predict(dmat, ntree_limit=ntree).astype(np.float32)


rows = []
for entry in index:
    data   = np.load(entry["cache_file"])
    ticker = entry["ticker"]
    vs, ve = int(entry["val_start"]), int(entry["val_end"])
    Xv     = data["X"][vs:ve].astype(np.float32)
    yv     = data["y"][vs:ve].astype(np.float32)
    dv     = pd.to_datetime(data["dates"][vs:ve])
    if len(Xv) == 0:
        continue
    pv = predict_probs_booster(booster, scaler.transform(Xv).astype(np.float32), best_ntree)
    rows.append(pd.DataFrame({"Date": dv, "ticker": ticker,
                               "y_true": yv.astype(np.int8), "prob_buy": pv}))

val_preds = pd.concat(rows, ignore_index=True)
val_preds["Date"]     = pd.to_datetime(val_preds["Date"]).dt.normalize()
val_preds["pred_buy"] = (val_preds["prob_buy"] >= SELECTED_THRESHOLD).astype(np.int8)

trades = val_preds[val_preds["pred_buy"] == 1].copy()
trades["is_success"] = (trades["y_true"] == 1).astype(np.int8)
trades["is_fail"]    = (trades["y_true"] == 0).astype(np.int8)

daily = (trades.groupby("Date")
               .agg(num_trades=("pred_buy", "size"),
                    num_success=("is_success", "sum"),
                    num_fail=("is_fail", "sum"))
               .reset_index().sort_values("Date"))
daily["pct_success"] = 100.0 * daily["num_success"] / daily["num_trades"].clip(lower=1)
daily["pct_fail"]    = 100.0 * daily["num_fail"]    / daily["num_trades"].clip(lower=1)

num_val_days    = int(val_preds["Date"].nunique())
overall_success = 100.0 * daily["num_success"].sum() / max(1, daily["num_trades"].sum())

print(f"Threshold : {SELECTED_THRESHOLD:.3f}")
print(f"Val days  : {num_val_days}  |  Total BUY trades: {int(daily['num_trades'].sum())}")
print(f"P(success | BUY): {overall_success:.2f}%")
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

wandb.log({
    "val_analysis/threshold":    SELECTED_THRESHOLD,
    "val_analysis/total_trades": int(daily["num_trades"].sum()),
    "val_analysis/pct_success":  overall_success,
    "val_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
})


# Test Data Analysis

In [ ]:

# ── Test Data Analysis: trades-per-day ────────────────────────────────────────

import numpy as np
import pandas as pd
import pickle
import joblib
import xgboost as xgb
import wandb

SELECTED_THRESHOLD = 0.9

with open(index_path, "rb") as f:
    index = pickle.load(f)
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

bundle     = joblib.load("best_model_xgb.pkl")
booster    = bundle["booster"]
best_ntree = int(bundle["best_ntree"])


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, ntree: int) -> np.ndarray:
    dmat = xgb.DMatrix(X)
    try:
        return booster.predict(dmat, iteration_range=(0, ntree)).astype(np.float32)
    except TypeError:
        return booster.predict(dmat, ntree_limit=ntree).astype(np.float32)


rows = []
for entry in index:
    data       = np.load(entry["cache_file"])
    ticker     = entry["ticker"]
    test_start = int(entry["test_start"])
    Xt = data["X"][test_start:].astype(np.float32)
    yt = data["y"][test_start:].astype(np.float32)
    dt = pd.to_datetime(data["dates"][test_start:])
    if len(Xt) == 0:
        continue
    pt = predict_probs_booster(booster, scaler.transform(Xt).astype(np.float32), best_ntree)
    rows.append(pd.DataFrame({"Date": dt, "ticker": ticker,
                               "y_true": yt.astype(np.int8), "prob_buy": pt}))

test_preds = pd.concat(rows, ignore_index=True)
test_preds["Date"]     = pd.to_datetime(test_preds["Date"]).dt.normalize()
test_preds["pred_buy"] = (test_preds["prob_buy"] >= SELECTED_THRESHOLD).astype(np.int8)

trades = test_preds[test_preds["pred_buy"] == 1].copy()
trades["is_success"] = (trades["y_true"] == 1).astype(np.int8)
trades["is_fail"]    = (trades["y_true"] == 0).astype(np.int8)

daily = (trades.groupby("Date")
               .agg(num_trades=("pred_buy", "size"),
                    num_success=("is_success", "sum"),
                    num_fail=("is_fail", "sum"))
               .reset_index().sort_values("Date"))
daily["pct_success"] = 100.0 * daily["num_success"] / daily["num_trades"].clip(lower=1)
daily["pct_fail"]    = 100.0 * daily["num_fail"]    / daily["num_trades"].clip(lower=1)

num_test_days   = int(test_preds["Date"].nunique())
overall_success = 100.0 * daily["num_success"].sum() / max(1, daily["num_trades"].sum())

print(f"Threshold  : {SELECTED_THRESHOLD:.3f}")
print(f"Test days  : {num_test_days}  |  Total BUY trades: {int(daily['num_trades'].sum())}")
print(f"P(success | BUY): {overall_success:.2f}%")
display(daily[["Date", "num_trades", "pct_success", "pct_fail", "num_success", "num_fail"]])

wandb.log({
    "test_analysis/threshold":    SELECTED_THRESHOLD,
    "test_analysis/total_trades": int(daily["num_trades"].sum()),
    "test_analysis/pct_success":  overall_success,
    "test_analysis/daily_table":  wandb.Table(dataframe=daily.reset_index(drop=True)),
})


In [ ]:
# don't execute the next cells
raise KeyboardInterrupt

# Real Life Test #3 :: using it to make a decision

In [ ]:
# ============================================================
# Real Life Testing (#3): scan ALL tickers at START OF DAY with OPEN price
# and print BUY/NO BUY using XGBoost (native Booster bundle)
#
# SCENARIO: Use at market open when you have:
# - Today's opening price
# - All previous complete bars (yesterday's full OHLCV and earlier)
#
# REQUIREMENTS (already in your notebook):
# - TICKERS (list of tickers)
# - WINDOW (int)  e.g. 30
# - INTERVAL (str) e.g. "1d"
# - BUY_THRESHOLD (float) e.g. 0.7
# - best_model_xgb.pkl saved as {"booster": Booster, "best_ntree": int}
# - scaler.pkl exists at .feature_cache_forward_return_w{WINDOW}/scaler.pkl
# ============================================================

import time
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
import joblib
import xgboost as xgb
from pathlib import Path
from zoneinfo import ZoneInfo

# ----------------------------
# CONFIG (using hyperparameters set at the top of the notebook)
# ----------------------------
PROB_THRESHOLD = BUY_THRESHOLD  # Use the BUY_THRESHOLD from config
LOOKBACK_DAYS  = max(WINDOW * 3, 180)  # Ensure enough lookback for window + warmup
CHUNK_SIZE     = 40
SLEEP_BETWEEN_CHUNKS = 0.5
INTERVAL = "1d"

# Display current hyperparameters being used
print(f"Using hyperparameters:")
print(f"  BUY_THRESHOLD:      {BUY_THRESHOLD}")
print(f"  PROFIT_THRESHOLD:   {PROFIT_THRESHOLD}")
print(f"  HORIZON_BARS:       {HORIZON_BARS}")
print(f"  WINDOW:             {WINDOW}")
print(f"  INTERVAL:           {INTERVAL}")
print(f"  LOOKBACK_DAYS:      {LOOKBACK_DAYS}")
print()

tz_market = ZoneInfo("America/New_York")

# For start-of-day trading: use today's date at market open time
# This ensures we get all previous complete bars (yesterday and before)
today_market = pd.Timestamp.now(tz_market).normalize()  # Today at 00:00
start_market  = today_market - pd.Timedelta(days=LOOKBACK_DAYS)
# IMPORTANT: To get data from today, you have to set end date to tomorrow!
end_market    = today_market + pd.Timedelta(days=1)  # Include today's data


def _to_naive_utc(ts: pd.Timestamp) -> pd.Timestamp:
    return ts.tz_convert("UTC").tz_localize(None) if ts.tz is not None else ts


start_naive_utc = _to_naive_utc(start_market)
end_naive_utc   = _to_naive_utc(end_market)


def predict_probs_booster(booster: xgb.Booster, X: np.ndarray, best_ntree: int) -> np.ndarray:
    """Predict probabilities from a native xgboost Booster (version-safe)."""
    dmat = xgb.DMatrix(X)
    try:
        probs = booster.predict(dmat, iteration_range=(0, best_ntree))
    except TypeError:
        probs = booster.predict(dmat, ntree_limit=best_ntree)
    return probs.astype(np.float32)


def _ema(s: pd.Series, span: int) -> pd.Series:
    return s.ewm(span=span, adjust=False, min_periods=span).mean()


def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def _true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    return pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)


def _atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = _true_range(high, low, close)
    return tr.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()


def _macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False, min_periods=signal).mean()
    macd_hist = macd - macd_signal
    return macd, macd_signal, macd_hist


def _stoch_k(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    ll = low.rolling(period, min_periods=period).min()
    hh = high.rolling(period, min_periods=period).max()
    denom = (hh - ll).replace(0.0, np.nan)
    return 100.0 * (close - ll) / denom


def _stoch_d(stoch_k: pd.Series, smooth: int = 3) -> pd.Series:
    return stoch_k.rolling(smooth, min_periods=smooth).mean()


def _cci(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 20) -> pd.Series:
    tp = (high + low + close) / 3.0
    sma = tp.rolling(n, min_periods=n).mean()
    md = tp.rolling(n, min_periods=n).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (tp - sma) / (0.015 * (md + 1e-12))


def _adx_dmi(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14):
    up_move   = high.diff()
    down_move = -low.diff()

    plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    prev_close = close.shift(1)
    tr = pd.concat(
    [
        (high - low).abs(),
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ],
    axis=1,
    ).max(axis=1)

    tr_smooth = tr.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    plus_dm_smooth  = pd.Series(plus_dm, index=high.index).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    minus_dm_smooth = pd.Series(minus_dm, index=high.index).ewm(alpha=1/n, adjust=False, min_periods=n).mean()

    plus_di  = 100.0 * (plus_dm_smooth  / (tr_smooth + 1e-12))
    minus_di = 100.0 * (minus_dm_smooth / (tr_smooth + 1e-12))

    dx  = 100.0 * ((plus_di - minus_di).abs() / ((plus_di + minus_di) + 1e-12))
    adx = dx.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return adx, plus_di, minus_di


def add_my_ta_features(df: pd.DataFrame) -> pd.DataFrame:
    """Trend/EMA/MACD/RSI/Stoch/CCI/ADX (causal)"""
    out = df.copy()
    c = out["Close"]
    h = out["High"]
    l = out["Low"]

    out["ema_10"] = _ema(c, 10)
    out["ema_20"] = _ema(c, 20)
    out["ema_50"] = _ema(c, 50)

    macd, macd_sig, macd_hist = _macd(c, 12, 26, 9)
    out["macd"] = macd
    out["macd_signal"] = macd_sig
    out["macd_hist"] = macd_hist

    out["rsi_14"] = _rsi(c, 14)

    stoch_k = _stoch_k(h, l, c, 14)
    out["stoch_k_14"] = stoch_k
    out["stoch_d_14"] = _stoch_d(stoch_k, 3)

    out["cci_20"] = _cci(h, l, c, 20)

    adx, plus_di, minus_di = _adx_dmi(h, l, c, 14)
    out["adx_14"] = adx
    out["plus_di_14"] = plus_di
    out["minus_di_14"] = minus_di

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def build_model_features_from_df(df_upto: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Build the SAME engineered feature set as training (no labels).
    
    SCENARIO: We BUY at market OPEN, only knowing today's OPEN price.
    Features use only PREVIOUS day's complete OHLCV + today's OPEN.
    
    FOR PRODUCTION: Pass in historical data with complete bars PLUS today's row
    with ONLY Open filled (High, Low, Close, Volume can be NaN or last known values).
    
    Expects columns: Date, Open, High, Low, Close, Volume.
    Returns DataFrame with columns ["Date", <features + lags>].
    """
    if "Date" not in df_upto.columns:
        raise ValueError("build_model_features_from_df expects a 'Date' column.")

    df = df_upto.copy()

    # Normalize and sort by Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    # Ensure numeric OHLCV
    for c in ["Open", "High", "Low", "Close", "Volume"]:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in df_upto for feature building.")
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # For production: last row might only have Open filled
    # We need at least Open for all rows
    df = df.dropna(subset=["Open"]).reset_index(drop=True)

    if len(df) < max(window + 2, 2):
        return pd.DataFrame(columns=["Date"])

    # =========================
    # Features: Only use data available at market OPEN
    # - All historical (t-1, t-2, ...) complete OHLCV
    # - Current day (t): ONLY the Open price (no High, Low, Close, Volume)
    # =========================
    
    # SHIFT all OHLCV data by 1 period to represent "previous" complete bars
    # The current row now represents what we knew BEFORE today opened
    df["prev_Open"] = df["Open"].shift(1)
    df["prev_High"] = df["High"].shift(1)
    df["prev_Low"] = df["Low"].shift(1)
    df["prev_Close"] = df["Close"].shift(1)
    df["prev_Volume"] = df["Volume"].shift(1)
    
    # Today's open is known (no shift needed for features that use it)
    df["today_Open"] = df["Open"]
    
    # All calculations below use PREVIOUS day's OHLCV (shifted data)
    # This ensures we only use information available before today's open
    
    # Basic per-bar returns / range (using PREVIOUS close)
    df["logret_1"] = np.log(df["prev_Close"] / df["prev_Close"].shift(1))
    df["ret_co"] = (df["prev_Close"] / df["prev_Open"]) - 1.0
    df["range_hl"] = (df["prev_High"] - df["prev_Low"]) / df["prev_Close"]

    # Gap from previous close to today's open
    df["gap_prevclose_to_open"] = (df["today_Open"] / df["prev_Close"]) - 1.0
    
    # Relative previous OHLC to close before that (scale-free price levels)
    close_t2 = df["prev_Close"].shift(1)  # Close from t-2
    df["open_rel_prevclose"] = (df["prev_Open"] / close_t2) - 1.0
    df["high_rel_prevclose"] = (df["prev_High"] / close_t2) - 1.0
    df["low_rel_prevclose"] = (df["prev_Low"] / close_t2) - 1.0

    # Candle anatomy of PREVIOUS bar (normalized)
    body = df["prev_Close"] - df["prev_Open"]
    denom_close = df["prev_Close"].replace(0.0, np.nan)
    df["body_size"] = body / denom_close
    upper_raw = df["prev_High"] - pd.concat([df["prev_Open"], df["prev_Close"]], axis=1).max(axis=1)
    lower_raw = pd.concat([df["prev_Open"], df["prev_Close"]], axis=1).min(axis=1) - df["prev_Low"]
    df["upper_wick_norm"] = upper_raw / denom_close
    df["lower_wick_norm"] = lower_raw / denom_close
    range_raw = (df["prev_High"] - df["prev_Low"]).replace(0.0, np.nan)
    df["body_to_range"] = body.abs() / range_raw
    df["clv"] = ((df["prev_Close"] - df["prev_Low"]) - (df["prev_High"] - df["prev_Close"])) / range_raw

    # ATR-based volatility (using PREVIOUS bars)
    df["atr14"] = _atr(df["prev_High"], df["prev_Low"], df["prev_Close"], period=14)
    df["atr14_norm"] = df["atr14"] / denom_close
    df["atr14_impulse_60"] = df["atr14_norm"] / df["atr14_norm"].rolling(60, min_periods=60).mean()

    # Rolling returns / momentum (using PREVIOUS close prices)
    for win in (3, 6, 12):
        roll = df["logret_1"].rolling(win, min_periods=win).sum()
        df[f"logret_sum_{win}"] = roll
        df[f"logret_mean_{win}"] = roll / float(win)

    # Simple N-bar price slopes (using PREVIOUS close)
    for win in (3, 6, 12):
        df[f"slope_close_{win}"] = (df["prev_Close"] - df["prev_Close"].shift(win)) / float(win)

    # Distance to rolling highs/lows (using PREVIOUS close)
    roll_max_40 = df["prev_Close"].rolling(40, min_periods=40).max()
    roll_min_20 = df["prev_Close"].rolling(20, min_periods=20).min()
    df["dist_to_HH_40"] = (df["prev_Close"] / roll_max_40) - 1.0
    df["dist_to_LL_20"] = (df["prev_Close"] / roll_min_20) - 1.0

    # Realized volatility (std of log returns from PREVIOUS closes)
    for win in (10, 20, 40):
        df[f"rv_{win}"] = df["logret_1"].rolling(win, min_periods=win).std()

    # Vol-of-vol
    df["rv_10_vol_20"] = df["rv_10"].rolling(20, min_periods=20).std()

    # Range expansion z-score (using PREVIOUS bars)
    re_mean = df["range_hl"].rolling(20, min_periods=20).mean()
    re_std = df["range_hl"].rolling(20, min_periods=20).std()
    df["range_expansion_z"] = (df["range_hl"] - re_mean) / re_std

    # Range position (using PREVIOUS bars)
    for win in (10, 20):
        roll_min_l = df["prev_Low"].rolling(win, min_periods=win).min()
        roll_max_h = df["prev_High"].rolling(win, min_periods=win).max()
        denom_range = (roll_max_h - roll_min_l).replace(0.0, np.nan)
        df[f"range_pos_{win}"] = (df["prev_Close"] - roll_min_l) / denom_range

    # Z-scores (using PREVIOUS close)
    close_mean_20 = df["prev_Close"].rolling(20, min_periods=20).mean()
    close_std_20 = df["prev_Close"].rolling(20, min_periods=20).std()
    df["z_close_20"] = (df["prev_Close"] - close_mean_20) / close_std_20

    logret_mean_20 = df["logret_1"].rolling(20, min_periods=20).mean()
    logret_std_20 = df["logret_1"].rolling(20, min_periods=20).std()
    df["z_logret_1_20"] = (df["logret_1"] - logret_mean_20) / logret_std_20

    # Parkinson volatility (using PREVIOUS bars)
    hl_ratio = (df["prev_High"] / df["prev_Low"]).replace({0.0: np.nan})
    parkinson_bar = (np.log(hl_ratio)) ** 2
    parkinson_const = 1.0 / (4.0 * np.log(2.0))
    df["parkinson_20"] = (parkinson_const * parkinson_bar.rolling(20, min_periods=20).mean()) ** 0.5

    # Garman-Klass volatility (using PREVIOUS bars)
    log_hl = np.log((df["prev_High"] / df["prev_Low"]).replace({0.0: np.nan}))
    log_co = np.log((df["prev_Close"] / df["prev_Open"]).replace({0.0: np.nan}))
    gk_var = 0.5 * (log_hl ** 2) - (2.0 * np.log(2.0) - 1.0) * (log_co ** 2)
    df["gk_vol_20"] = (gk_var.rolling(20, min_periods=20).mean()) ** 0.5

    # Volume features (using PREVIOUS bars only)
    df["dollar_vol_log"] = np.log1p(df["prev_Close"] * df["prev_Volume"])
    for win in (10, 20, 40):
        vol_ma = df["prev_Volume"].rolling(win, min_periods=win).mean()
        df[f"vol_rel_{win}"] = df["prev_Volume"] / vol_ma

    df["pv_agree_20"] = df["logret_1"] * df["vol_rel_20"]

    # OBV-like (using PREVIOUS close)
    sign_close = np.sign(df["prev_Close"].diff().fillna(0.0))
    df["obv"] = (sign_close * df["prev_Volume"]).cumsum()
    df["obv_change_20"] = df["obv"] - df["obv"].shift(20)

    # Create a dataframe with PREVIOUS complete bars for TA indicators
    df_prev = pd.DataFrame({
        "High": df["prev_High"],
        "Low": df["prev_Low"],
        "Close": df["prev_Close"],
        "Open": df["prev_Open"]
    })
    
    # Trend filters from EMAs / MACD (using PREVIOUS closes)
    df_prev = add_my_ta_features(df_prev)
    for col in df_prev.columns:
        if col not in ["High", "Low", "Close", "Open"]:
            df[col] = df_prev[col]

    for span in (10, 20, 50):
        ema_col = f"ema_{span}"
        df[f"dist_to_ema_{span}"] = (df["prev_Close"] / df[ema_col]) - 1.0
        df[f"ema_slope_{span}"] = df[ema_col] - df[ema_col].shift(1)

    # Interaction features
    df["trend_vol_20"] = df["dist_to_ema_20"] * df["rv_20"]
    df["mom_vol_6_20"] = df["logret_sum_6"] * df["vol_rel_20"]
    df["meanrev_vol_20"] = df["z_close_20"] / (df["rv_20"] + 1e-12)

    # VWAP 20 (using PREVIOUS bars only)
    tp = (df["prev_High"] + df["prev_Low"] + df["prev_Close"]) / 3.0
    vol_roll_20 = df["prev_Volume"].rolling(20, min_periods=20).sum()
    vwap_num_20 = (tp * df["prev_Volume"]).rolling(20, min_periods=20).sum()
    df["vwap_20"] = (vwap_num_20 / vol_roll_20).replace([np.inf, -np.inf], np.nan)
    df["dist_to_vwap_20"] = (df["prev_Close"] / df["vwap_20"]) - 1.0

    feat_now = [
    # Relative OHLC / gap (core for open trading)
    "open_rel_prevclose","high_rel_prevclose","low_rel_prevclose",
    "gap_prevclose_to_open",

    # Returns / range
    "logret_1","ret_co","range_hl",
    "logret_sum_3","logret_sum_6","logret_sum_12",
    # (often better than logret_mean_*; can drop means)
    "z_logret_1_20",

    # Candle anatomy
    "body_size","upper_wick_norm","lower_wick_norm","body_to_range","clv",

    # Volatility / risk regime (your strongest block)
    "atr14_norm","atr14_impulse_60",
    "gk_vol_20",
    "rv_10","rv_20","rv_40","rv_10_vol_20",
    "range_expansion_z",

    # Location in range / mean reversion
    "range_pos_10","range_pos_20","z_close_20",
    "dist_to_HH_40","dist_to_LL_20",

    # Volume confirmation
    "dollar_vol_log","vol_rel_10","vol_rel_20","vol_rel_40","pv_agree_20",
    "obv","obv_change_20",

    # Oscillators / trend (keep only the ones that tend to matter)
    "macd_hist","rsi_14","adx_14","cci_20","stoch_k_14","stoch_d_14",

    # Interactions you already had (2 of them were useful)
    "mom_vol_6_20","meanrev_vol_20",
    ]

    # Lag features for WINDOW (lags 1..window-1)
    lagged = [df[feat_now].shift(lag).add_suffix(f"_lag_{lag}") for lag in range(1, window)]
    Xdf = pd.concat([df[feat_now]] + lagged, axis=1)

    # Note: Volume features are already log-transformed in dollar_vol_log
    # No need for additional log1p transform

    Xdf = Xdf.replace([np.inf, -np.inf], np.nan)

    out = pd.concat([df[["Date"]], Xdf], axis=1)
    out = out.dropna(axis=0).reset_index(drop=True)
    return out


# ============================================================
# LOAD SCALER + XGB BOOSTER BUNDLE
# ============================================================
scaler_path = Path(f".feature_cache_forward_return_w{WINDOW}") / "scaler.pkl"
model_path  = Path("best_model_xgb.pkl")

assert scaler_path.exists(), f"Missing scaler at: {scaler_path}"
assert model_path.exists(),  f"Missing XGBoost model bundle at: {model_path}"

with open(scaler_path, "rb") as f:
    scaler_loaded = pickle.load(f)

bundle = joblib.load(model_path)
assert isinstance(bundle, dict) and "booster" in bundle and "best_ntree" in bundle, \
    "best_model_xgb.pkl must be a dict bundle: {'booster': Booster, 'best_ntree': int}"

booster = bundle["booster"]
best_ntree = int(bundle["best_ntree"])

# de-dup tickers while preserving order
seen = set()
TICKERS_UNIQ = []
for t in TICKERS:
    if t not in seen:
        TICKERS_UNIQ.append(t)
        seen.add(t)

def _extract_one_ticker_df(df_all: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Extract a single-ticker OHLCV frame from yf.download multi-ticker output."""
    if df_all is None or df_all.empty:
        return pd.DataFrame()

    cols = df_all.columns

    # common: df_all is MultiIndex with [Ticker, Field] or [Field, Ticker]
    if isinstance(cols, pd.MultiIndex) and ticker in cols.get_level_values(0):
        d = df_all[ticker].copy()
        need = ["Open", "High", "Low", "Close", "Volume"]
        if not set(need).issubset(d.columns):
            return pd.DataFrame()
        return d[need].copy()

    if isinstance(cols, pd.MultiIndex) and ticker in cols.get_level_values(-1):
        need = ["Open", "High", "Low", "Close", "Volume"]
        out = pd.DataFrame(index=df_all.index)
        for fld in need:
            if (fld, ticker) in cols:
                out[fld] = df_all[(fld, ticker)]
            elif (ticker, fld) in cols:
                out[fld] = df_all[(ticker, fld)]
            else:
                return pd.DataFrame()
        return out

    # single ticker case
    if not isinstance(cols, pd.MultiIndex):
        need = ["Open", "High", "Low", "Close", "Volume"]
        if set(need).issubset(df_all.columns):
            return df_all[need].copy()

    return pd.DataFrame()

def predict_prob_buy_from_history(df_upto: pd.DataFrame, today_open_price: float = None) -> dict:
    """
    Compute BUY probability at START OF DAY using only opening price.
    
    Args:
        df_upto: Complete historical bars (yesterday and before)
        today_open_price: Today's opening price (if available)
    
    For start-of-day trading:
    - df_upto contains all COMPLETE bars up to yesterday
    - today_open_price is today's opening price
    - We create a partial row for "today" with only Open filled
    """
    if df_upto is None or df_upto.empty:
        return {"ok": False, "reason": "no history"}

    # Get the last complete bar (yesterday)
    last_complete_time = df_upto.index[-1]
    last_complete_row = df_upto.iloc[-1]

    # Prepare dataframe with historical data
    tmp = df_upto.reset_index().rename(columns={df_upto.index.name or "index": "Date"})
    
    # If we have today's opening price, add it as a partial row
    if today_open_price is not None:
        # Create today's row with only Open price (other fields will be NaN)
        today_date = last_complete_time + pd.Timedelta(days=1)
        today_row = pd.DataFrame({
            "Date": [today_date],
            "Open": [today_open_price],
            "High": [np.nan],  # Not yet known at market open
            "Low": [np.nan],   # Not yet known at market open
            "Close": [np.nan], # Not yet known at market open
            "Volume": [np.nan] # Not yet known at market open
        })
        tmp = pd.concat([tmp, today_row], ignore_index=True)
    
    feats = build_model_features_from_df(tmp, WINDOW)
    if feats.empty:
        return {"ok": False, "reason": "features NaN (insufficient warmup)"}

    last = feats.iloc[-1]
    X_row = last.drop(labels=["Date"]).to_numpy(dtype=np.float32, copy=False).reshape(1, -1)

    Xs = scaler_loaded.transform(X_row).astype(np.float32, copy=False)
    prob_buy = float(predict_probs_booster(booster, Xs, best_ntree)[0])

    return {
        "ok": True,
        "bar_used_market": last_complete_time,
        "prob_buy": prob_buy,
        "cutoff_open": float(last_complete_row["Open"]),
        "cutoff_high": float(last_complete_row["High"]),
        "cutoff_low": float(last_complete_row["Low"]),
        "cutoff_close": float(last_complete_row["Close"]),
        "cutoff_volume": float(last_complete_row["Volume"]),
        "today_open": today_open_price if today_open_price is not None else None,
    }

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

results = []
errors = []

print(f"Scanning {len(TICKERS_UNIQ)} tickers at START OF DAY")
print(f"Market timezone: {tz_market}, today: {today_market}")
print(f"Downloading history from {start_market} to {end_market} (market time) ...")
print(f"This will get ALL previous complete bars plus TODAY'S opening price\n")

for chunk in chunks(TICKERS_UNIQ, CHUNK_SIZE):
    try:
        df_all = yf.download(
            tickers=chunk,
            start=start_naive_utc,
            end=end_naive_utc,
            interval=INTERVAL,
            auto_adjust=False,
            progress=False,
            group_by="ticker",
            threads=True,
        )
    except Exception as e:
        for t in chunk:
            errors.append({"ticker": t, "error": f"download failed: {e}"})
        time.sleep(SLEEP_BETWEEN_CHUNKS)
        continue

    if df_all is None or df_all.empty:
        for t in chunk:
            errors.append({"ticker": t, "error": "empty download"})
        time.sleep(SLEEP_BETWEEN_CHUNKS)
        continue

    # yfinance daily index is often naive; keep it in market tz for the cutoff filter
    idx = pd.to_datetime(df_all.index, errors="coerce")
    df_all = df_all.copy()
    df_all.index = idx
    df_all = df_all[~df_all.index.isna()]

    if df_all.index.tz is None:
        df_all.index = df_all.index.tz_localize(tz_market)
    else:
        df_all.index = df_all.index.tz_convert(tz_market)

    for t in chunk:
        d = _extract_one_ticker_df(df_all, t)
        if d.empty:
            errors.append({"ticker": t, "error": "no usable OHLCV in download"})
            continue

        d = d.sort_index()
        
        # Split into complete bars (yesterday and before) and today's data
        # Complete bars: have all OHLCV data
        d_complete = d.dropna(subset=["Open", "High", "Low", "Close", "Volume"])
        
        # Check if we have today's opening price
        # Today's bar might be incomplete (only Open available)
        today_bars = d[d.index >= today_market]
        today_open_price = None
        
        if not today_bars.empty and not pd.isna(today_bars.iloc[0]["Open"]):
            # We have today's opening price!
            today_open_price = float(today_bars.iloc[0]["Open"])
        
        # Check if today's opening price is available
        if today_open_price is None:
            print(f"⚠️  WARNING: {t} - Today's opening price not found. Market may not be open yet or data is incomplete. Skipping.")
            errors.append({"ticker": t, "error": "today's opening price not available"})
            continue
        
        # Use only complete historical bars (yesterday and before)
        d_history = d_complete[d_complete.index < today_market]
        
        if d_history.empty:
            errors.append({"ticker": t, "error": "no complete historical bars"})
            continue

        res = predict_prob_buy_from_history(d_history, today_open_price)
        if not res["ok"]:
            errors.append({"ticker": t, "error": res["reason"]})
            continue

        p = res["prob_buy"]
        decision = "BUY" if p >= PROB_THRESHOLD else "NO BUY"

        result_data = {
            "ticker": t,
            "bar_used_market": res["bar_used_market"],
            "prob_buy": p,
            "prob_buy_%": 100.0 * p,
            "decision": decision,
            "yesterday_close": res["cutoff_close"],
        }
        
        # Add today's opening price if available
        if today_open_price is not None:
            result_data["today_open"] = today_open_price
            result_data["gap_%"] = 100.0 * (today_open_price / res["cutoff_close"] - 1.0)
        
        results.append(result_data)

    time.sleep(SLEEP_BETWEEN_CHUNKS)

df_res = pd.DataFrame(results).sort_values(["decision", "prob_buy"], ascending=[False, False]).reset_index(drop=True)
df_err = pd.DataFrame(errors)
if not df_err.empty and "ticker" in df_err.columns:
    df_err = df_err.sort_values("ticker").reset_index(drop=True)

print("\n================ RESULTS ================")
print(f"OK: {len(df_res)} tickers | Errors: {len(df_err)} tickers")
print(f"\nNOTE: This scan is designed for START OF DAY trading.")
print(f"      Features use YESTERDAY's complete bar + TODAY's opening price (if available).")
print(f"      If 'today_open' column is present, the gap% shows overnight gap.\n")
display(df_res)

if not df_err.empty:
    print("\n================ ERRORS (top 50) ================")
    display(df_err.head(50))

out_csv = f"scan_start_of_day_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_res.to_csv(out_csv, index=False)
print(f"\nSaved results to: {out_csv}")
